In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from transformers import BertTokenizer, BertModel
import torch
from tqdm import tqdm

/usr/local/lib/python3.12/dist-packages/torch_xla/experimental/gru.py:113: SyntaxWarning: invalid escape sequence '\_'
  * **h_n**: tensor of shape :math:`(D * \text{num\_layers}, H_{out})` or
/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:82: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [3]:

def cleaning_outliers_dataset():
    tqdm.pandas()

    # Load bert
    tokenizer = BertTokenizer.from_pretrained("bert-base-multilingual-cased")
    model = BertModel.from_pretrained("bert-base-multilingual-cased")
    model.eval()

    def process(df, texts, keterangan):
        # Encoding bert
        def get_bert_embedding(text):
            inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
            with torch.no_grad():
                outputs = model(**inputs)
            return outputs.last_hidden_state[:, 0, :].squeeze().numpy()

        embeddings = []
        for text in tqdm(texts, desc=f"Extraction BERT embeddings ({keterangan})"):
            try:
                emb = get_bert_embedding(text)
                embeddings.append(emb)
            except:
                embeddings.append(np.zeros(768))

        embeddings = [get_bert_embedding(text) for text in tqdm(texts, desc=f"BERT Embeddings ({keterangan})")]
        embeddings = np.array(embeddings)

        # PCA
        pca = PCA(n_components=min(50, embeddings.shape[1]))
        X_reduced = pca.fit_transform(embeddings)

        # Deteksi outlier dengan isolation forest
        iso = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
        outlier_preds = iso.fit_predict(X_reduced)

        df = df.copy()

        # Label: 1 = normal, -1 = outlier
        df['outlier'] = outlier_preds

        outlier_df = df[df['outlier'] == -1]
        normal_df = df[df['outlier'] != -1]

        return outlier_df, normal_df


    def inggris():
        df = pd.read_csv("/content/drive/My Drive/dataset/3datasentimen_clean_without_outlier_and_lowertext34.csv")

        # Filter inggris
        df = df[df['bahasa'] == 'En'].reset_index(drop=True)
        texts = df['text'].tolist()
        ket = "inggris"

        outlier_df, normal_df = process(df=df, texts=texts, keterangan=ket)

        # Simpan
        outlier_df.to_csv("/content/drive/My Drive/dataset/4datasentimen_outlier_ing_googlecolaboutlier34.csv", index=False)
        normal_df.to_csv("/content/drive/My Drive/dataset/4datasentimen_all_clean_ing_googlecolaboutlier34.csv", index=False)

    inggris()
    # proses_gabung()


In [4]:
cleaning_outliers_dataset()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

BERT Embeddings (inggris): 100%|██████████| 7269/7269 [02:14<00:00, 54.22it/s]


In [6]:
df = pd.read_csv(f"/content/drive/My Drive/dataset/4datasentimen_all_clean_ing_googlecolaboutlier34.csv", sep=",")
print(df["combine_labelbahasa"].value_counts())

combine_labelbahasa
En_0    4299
En_1    2606
Name: count, dtype: int64
